In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, col, coalesce, when

spark=(
    SparkSession
    .builder
    .appName("CSV-ETL")
    .master("local[*]")
    .getOrCreate()
)
spark

## Remove Duplicates

In [14]:
df = spark.read.format('csv').option("header", "true").option("inferschema", "true").load("vendor_sales.csv")
df.show()

+--------+-----------+----------+------+-------------+
|order_id|customer_id|order_date|amount|      country|
+--------+-----------+----------+------+-------------+
|       0|        191|15-01-2026|  NULL|           US|
|       1|        145|2026/01/11| 168.0|           US|
|       2|        168|2026-01-03| 770.0|United States|
|       3|        170|05-02-2026|  NULL|         U.S.|
|       4|        189|06-02-2026| 417.0|          USA|
|       5|        154|01-02-2026| 779.0|         U.S.|
|       6|        185|21-02-2026| 132.0|         U.S.|
|       7|        199|2026-01-28|  NULL|         U.S.|
|       8|        136|2026/02/13|  NULL|          USA|
|       9|        110|03-02-2026| 442.0|          USA|
|      10|        191|2026/01/12|  NULL|          USA|
|      11|        140|11-02-2026|  NULL|           US|
|      12|        132|2026/01/29| 281.0|         U.S.|
|      13|        169|2026/01/11| 404.0|         U.S.|
|      14|        176|2026-02-19| 634.0|           US|
|      15|

In [15]:
df.count()

2100

In [16]:
df.distinct().count()

2000

In [17]:
df = df.dropDuplicates()

In [18]:
df.count()

2000

In [19]:
df.distinct().count()

2000

# standardize Date Format

In [20]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- country: string (nullable = true)



In [21]:
df.show()

+--------+-----------+----------+------+-------------+
|order_id|customer_id|order_date|amount|      country|
+--------+-----------+----------+------+-------------+
|     482|        115|2026-02-07| 822.0|          USA|
|    1218|        119|2026-02-03| 821.0|           US|
|    1862|        173|22-02-2026| 939.0|           US|
|     167|        137|2026-01-19| 899.0|         U.S.|
|     497|        147|2026-01-25| 580.0|          USA|
|     745|        184|2026/02/24| 909.0|         U.S.|
|    1110|        193|03-02-2026| 822.0|United States|
|    1165|        111|2026/02/13| 523.0|          USA|
|    1366|        103|2026-02-07| 998.0|United States|
|    1461|        172|01-02-2026| 397.0|          USA|
|    1724|        137|2026/01/31| 634.0|         U.S.|
|    1902|        182|2026-01-21| 282.0|          USA|
|      60|        139|2026-02-04| 225.0|           US|
|      81|        177|03-01-2026| 536.0|United States|
|     806|        125|09-01-2026| 638.0|          USA|
|     903|

In [22]:
df = df.withColumn("order_date", coalesce(
        to_date(col("order_date").cast("string"), "yyyy-MM-dd"),
        to_date(col("order_date").cast("string"), "dd-MM-yyyy"),
        to_date(col("order_date").cast("string"), "yyyy/MM/dd")
    ))

In [23]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- country: string (nullable = true)



# Normalization of countries

In [24]:
df_country=df.withColumn("country",when(col("country").isin(["US", "USA", "United States", "U.S."]),"USA").otherwise(col("country")))
df_country.show()

+--------+-----------+----------+------+-------+
|order_id|customer_id|order_date|amount|country|
+--------+-----------+----------+------+-------+
|     482|        115|2026-02-07| 822.0|    USA|
|    1218|        119|2026-02-03| 821.0|    USA|
|    1862|        173|2026-02-22| 939.0|    USA|
|     167|        137|2026-01-19| 899.0|    USA|
|     497|        147|2026-01-25| 580.0|    USA|
|     745|        184|2026-02-24| 909.0|    USA|
|    1110|        193|2026-02-03| 822.0|    USA|
|    1165|        111|2026-02-13| 523.0|    USA|
|    1366|        103|2026-02-07| 998.0|    USA|
|    1461|        172|2026-02-01| 397.0|    USA|
|    1724|        137|2026-01-31| 634.0|    USA|
|    1902|        182|2026-01-21| 282.0|    USA|
|      60|        139|2026-02-04| 225.0|    USA|
|      81|        177|2026-01-03| 536.0|    USA|
|     806|        125|2026-01-09| 638.0|    USA|
|     903|        195|2026-03-02| 245.0|    USA|
|    1153|        111|2026-01-24| 603.0|    USA|
|    1161|        16

In [25]:
df.show()

+--------+-----------+----------+------+-------------+
|order_id|customer_id|order_date|amount|      country|
+--------+-----------+----------+------+-------------+
|     482|        115|2026-02-07| 822.0|          USA|
|    1218|        119|2026-02-03| 821.0|           US|
|    1862|        173|2026-02-22| 939.0|           US|
|     167|        137|2026-01-19| 899.0|         U.S.|
|     497|        147|2026-01-25| 580.0|          USA|
|     745|        184|2026-02-24| 909.0|         U.S.|
|    1110|        193|2026-02-03| 822.0|United States|
|    1165|        111|2026-02-13| 523.0|          USA|
|    1366|        103|2026-02-07| 998.0|United States|
|    1461|        172|2026-02-01| 397.0|          USA|
|    1724|        137|2026-01-31| 634.0|         U.S.|
|    1902|        182|2026-01-21| 282.0|          USA|
|      60|        139|2026-02-04| 225.0|           US|
|      81|        177|2026-01-03| 536.0|United States|
|     806|        125|2026-01-09| 638.0|          USA|
|     903|

# Fill missing amounts with 0

In [26]:
df.show()

+--------+-----------+----------+------+-------------+
|order_id|customer_id|order_date|amount|      country|
+--------+-----------+----------+------+-------------+
|     482|        115|2026-02-07| 822.0|          USA|
|    1218|        119|2026-02-03| 821.0|           US|
|    1862|        173|2026-02-22| 939.0|           US|
|     167|        137|2026-01-19| 899.0|         U.S.|
|     497|        147|2026-01-25| 580.0|          USA|
|     745|        184|2026-02-24| 909.0|         U.S.|
|    1110|        193|2026-02-03| 822.0|United States|
|    1165|        111|2026-02-13| 523.0|          USA|
|    1366|        103|2026-02-07| 998.0|United States|
|    1461|        172|2026-02-01| 397.0|          USA|
|    1724|        137|2026-01-31| 634.0|         U.S.|
|    1902|        182|2026-01-21| 282.0|          USA|
|      60|        139|2026-02-04| 225.0|           US|
|      81|        177|2026-01-03| 536.0|United States|
|     806|        125|2026-01-09| 638.0|          USA|
|     903|

In [27]:
from pyspark.sql.functions import col, sum, when, lit

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()


+--------+-----------+----------+------+-------+
|order_id|customer_id|order_date|amount|country|
+--------+-----------+----------+------+-------+
|       0|          0|         0|   989|      0|
+--------+-----------+----------+------+-------+



In [28]:
df = df.withColumn(
    "amount",
    when(col("amount").isNull(), lit(0))  
    .otherwise(col("amount"))             
)

In [29]:
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+--------+-----------+----------+------+-------+
|order_id|customer_id|order_date|amount|country|
+--------+-----------+----------+------+-------+
|       0|          0|         0|     0|      0|
+--------+-----------+----------+------+-------+



# Output cleaned_sales.csv

In [30]:
df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("cleaned_sales.csv")